In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_page_master = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/page_master/page_master.csv")
)

print("Rows:", df_page_master.count())
print("Columns:", len(df_page_master.columns))

display(df_page_master.limit(10))


StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 3, Finished, Available, Finished, False)

Rows: 500
Columns: 7


SynapseWidget(Synapse.DataFrame, 19359d09-93b5-4749-98a1-325c8ef4ce0b)

In [2]:
print("Duplicate page IDs:")

duplicate_page_ids = (
    df_page_master
    .groupBy("page_id")
    .count()
    .filter(F.col("count") > 1)
)

print(duplicate_page_ids.count())

print("\nPage types:")

display(
    df_page_master
    .groupBy("page_type")
    .count()
    .orderBy("page_type")
)

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 4, Finished, Available, Finished, False)

Duplicate page IDs:
0

Page types:


SynapseWidget(Synapse.DataFrame, 1e577788-56d4-40bb-bbf2-738c994830da)

In [3]:
print("Active flag:")

display(
    df_page_master
    .groupBy("active_flag")
    .count()
    .orderBy("active_flag")
)

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 5, Finished, Available, Finished, False)

Active flag:


SynapseWidget(Synapse.DataFrame, bdf10db6-426c-4abe-b01f-b19d25711b09)

In [4]:
print("Null product IDs:")

print(
    df_page_master
    .filter(
        F.col("product_id").isNull() |
        (F.trim(F.col("product_id")) == "")
    )
    .count()
)

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 6, Finished, Available, Finished, False)

Null product IDs:
180


In [5]:
df_page_dim = (
    df_page_master
    .withColumn("page_id", F.upper(F.trim(F.col("page_id"))))
    .withColumn("page_url", F.trim(F.col("page_url")))
    .withColumn("page_type", F.initcap(F.trim(F.col("page_type"))))
    .withColumn("product_id", F.upper(F.trim(F.col("product_id"))))
    .withColumn("page_category", F.trim(F.col("page_category")))
    .withColumn("active_flag", F.trim(F.col("active_flag")))
)

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 7, Finished, Available, Finished, False)

In [6]:
print("Final page rows:", df_page_dim.count())
print("Final page columns:", len(df_page_dim.columns))

print("\nDuplicate page IDs:")

print(
    df_page_dim
    .groupBy("page_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nPage types:")

display(
    df_page_dim
    .groupBy("page_type")
    .count()
    .orderBy("page_type")
)

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 8, Finished, Available, Finished, False)

Final page rows: 500
Final page columns: 7

Duplicate page IDs:
0

Page types:


SynapseWidget(Synapse.DataFrame, a5674426-51d0-4a3a-b0f1-5fb6f7fff3c1)

In [7]:
(
    df_page_dim.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_page")
)

print("Silver dim_page table created successfully.")

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 9, Finished, Available, Finished, False)

Silver dim_page table created successfully.


In [8]:
df_silver_page = spark.read.table("dim_page")

print("Silver dim_page rows:", df_silver_page.count())
print("Silver dim_page columns:", len(df_silver_page.columns))

display(df_silver_page.limit(10))

StatementMeta(, 0b1bea1b-6fa9-47fa-875a-f67423a21a20, 10, Finished, Available, Finished, False)

Silver dim_page rows: 500
Silver dim_page columns: 7


SynapseWidget(Synapse.DataFrame, 60bab143-5e22-4475-83a8-736ee381eea8)